In [ ]:
!pip install -U -q bitsandbytes accelerate transformers

In [ ]:
!pip install vllm

In [ ]:
!nvidia-smi

In [ ]:
from vllm import LLM, SamplingParams
import json
import os

# --- 1. НАСТРОЙКИ ---
# Убедись, что путь к модели верный или укажи название с HF
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct" 
INPUT_FILE = "/kaggle/input/datasets/kirillsilantev/dostoevsky/dostoevsky_blocks_for_neutralization-2.json"
OUTPUT_FILE = "dostoevsky_parallel_final_v2.json"

# Системный промпт, который заставит Llama быть точным редактором
SYSTEM_PROMPT = (
    "Ты — профессиональный редактор и лингвист. Перескажи предложенный текст Достоевского "
    "максимально сухим, современным и информативным языком. Сохрани всех персонажей, их имена, "
    "последовательность действий и ключевые факты. Удали авторские отступления, архаизмы, "
    "сложные эпитеты и психологические метафоры. Твой ответ должен быть связным текстом."
)

# --- 2. ЗАГРУЗКА ДАННЫХ ---
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Формируем промпты в формате Llama-3
prompts = []
for item in dataset:
    msg = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Текст для пересказа:\n{item['original_fragment']}"}
    ]
    # Используем шаблон Llama-3 для корректной работы инструкций
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{item['original_fragment']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    prompts.append(prompt)

# --- 3. ЗАПУСК ГЕНЕРАЦИИ ---
# Используем tensor_parallel_size=2 для твоих двух P100
llm = LLM(
    model=MODEL_NAME, 
    tensor_parallel_size=2, 
    max_model_len=2048,
    trust_remote_code=True
)

# Настройки для точности: низкая температура = меньше галлюцинаций
sampling_params = SamplingParams(
    temperature=0.0, 
    max_tokens=1024,
    repetition_penalty=1.05
)

print(f"Начинаю генерацию для {len(prompts)} блоков...")
outputs = llm.generate(prompts, sampling_params)

# --- 4. СОХРАНЕНИЕ ---
final_results = []
for i, out in enumerate(outputs):
    neutral_text = out.outputs[0].text.strip()
    
    # Сохраняем все метаданные и добавляем нейтральный прокси
    entry = dataset[i]
    entry["neutral_proxy"] = neutral_text
    final_results.append(entry)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=4)

print(f"Готово! Файл сохранен: {OUTPUT_FILE}")

In [ ]:
/kaggle/input/datasets/kirillsilantev/dostoevsky/dostoevsky_blocks_for_neutralization-2.json

import json

#загружаем сохраненный результат
output_path = '/kaggle/working/dostoevsky_parallel_dataset.json'
with open(output_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

for i, item in enumerate(results):
    print(f"фрагмент {i+1}, скор: {item['topic_fidelity']}")
    print("-" * 30)
    
    print("оригинал:")
    print(item['original_fragment'])
    
    print("\nнейтральный:")
    print(item['neutral_proxy'])
    print(f"{'-'*100}\n")

In [ ]:
import torch
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
# 1. Настройки батчинга
BATCH_SIZE = 8  # Можно попробовать 16, если хватит VRAM
INPUT_FILE = '/kaggle/input/datasets/kirillsilantev/dostoevsky-fragments/dostoevsky_original_fragments1.json'
OUTPUT_FILE = '/kaggle/working/dostoevsky_parallel_dataset_final.json'

# Настройка токенайзера для батчинга
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Подготовка данных
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

final_data = []

# 3. Ускоренный цикл обработки
model.eval()
for i in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Batch Processing"):
    batch = dataset[i : i + BATCH_SIZE]
    
    # Формируем промпты для всей пачки
    prompts = []
    for item in batch:
        messages = [
            {"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
            {"role": "user", "content": f"Текст для упрощения: {item['original_fragment']}"}
        ]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    
    # Токенизация всей пачки сразу
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
    
    # Генерация для всей пачки
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            generation_config=generation_config,
            max_new_tokens=256,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Декодирование результатов
    for j, item in enumerate(batch):
        # Извлекаем только ответ ассистента, обрезая входной промпт
        generated_text = tokenizer.decode(output_ids[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        item['neutral_proxy'] = generated_text
        final_data.append(item)

# 4. Массовая оценка Topic Fidelity (так быстрее, чем в цикле)
print("Начинаю массовую оценку сходства...")
originals = [item['original_fragment'] for item in final_data]
neutrals = [item['neutral_proxy'] for item in final_data]

# Эмбеддер сам эффективно использует GPU для батчей
emb_orig = embedder.encode(originals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)
emb_neut = embedder.encode(neutrals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)

# Считаем косинусное сходство попарно
cosine_scores = util.cos_sim(emb_orig, emb_neut).diagonal().tolist()

for i, score in enumerate(cosine_scores):
    final_data[i]['topic_fidelity'] = round(score, 4)
    final_data[i]['is_valid'] = score > 0.75

# 5. Сохранение
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print(f"Готово! Средний скор по всему датасету: {sum(cosine_scores)/len(cosine_scores):.4f}")

saiga не работает в параллельных потоках, для этого нужна LLama с vLLM, поэтому дальше работаю с LLama

In [ ]:
# Ставим конкретную стабильную версию
!pip uninstall -y vllm flashinfer
!pip install -q vllm==0.6.3 sentence-transformers accelerate

In [ ]:
!pip install --upgrade vllm

In [ ]:
import os
import json
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# --- 1. ОЧИСТКА ---
gc.collect()
torch.cuda.empty_cache()

# --- 2. НАСТРОЙКИ ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
INPUT_FILE = "/kaggle/input/datasets/kirillsilantev/dostoevsky/dostoevsky_blocks_for_neutralization-2.json"
OUTPUT_FILE = "/kaggle/working/dostoevsky_parallel_final_v2.json"

SYSTEM_PROMPT = (
    "Ты — профессиональный русский редактор. Перескажи текст Достоевского "
    "исключительно на РУССКОМ языке. Сохрани всех персонажей и факты. "
    "Пиши сухим стилем, без архаизмов и пафоса."
)

# --- 3. ЗАГРУЗКА МОДЕЛЕЙ ---
print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Важно для батчинга:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" 

print("Загрузка модели Qwen на 2x T4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto", # Распределит слои между GPU 0 и 1
    trust_remote_code=True
)

# Оценщик сходства (на вторую карту, чтобы разгрузить первую)
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device="cuda:1")

# --- 4. ЗАГРУЗКА ДАННЫХ ---
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# --- 5. ГЕНЕРАЦИЯ ---
BATCH_SIZE = 4 # Начни с 4, если VRAM позволит — подними до 8
results = []

print(f"Старт обработки: {len(dataset)} фрагментов...")

# Используем модель напрямую для лучшего контроля скорости
model.eval()

for i in range(0, len(dataset), BATCH_SIZE):
    batch = dataset[i : i + BATCH_SIZE]
    prompts = []
    
    for item in batch:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item['original_fragment']}
        ]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    
    # Токенизация батча
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512, # Ограничил до 512, обычно для пересказа хватает
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Декодируем только новые токены (отрезаем входной промпт)
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    
    original_texts = [item['original_fragment'] for item in batch]
    
    # Расчет сходства
    emb_orig = embedder.encode(original_texts, convert_to_tensor=True, show_progress_bar=False)
    emb_neut = embedder.encode(generated_texts, convert_to_tensor=True, show_progress_bar=False)
    cosine_scores = util.cos_sim(emb_orig, emb_neut).diagonal().tolist()
    
    # Вывод и сборка
    for j in range(len(batch)):
        score = round(cosine_scores[j], 4)
        print(f"[{i+j}] Score: {score} | Text: {generated_texts[j][:60]}...")
        
        res = batch[j]
        res['neutral_proxy'] = generated_texts[j]
        res['topic_fidelity'] = score
        res['is_valid'] = score > 0.75
        results.append(res)
        
    # Периодическое сохранение (чтобы не потерять данные при падении)
    if i % (BATCH_SIZE * 5) == 0:
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)

# --- 6. ФИНАЛЬНОЕ СОХРАНЕНИЕ ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nВсе готово! Итоговый файл: {OUTPUT_FILE}")

2026-03-01 16:37:47.401030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772383067.592396     187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772383067.651182     187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772383068.093543     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772383068.093577     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772383068.093580     187 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Старт обработки: 2052 фрагментов...
[0] Score: 0.6884 | Text: Начиная описание героя своего романа, Алексея Федоровича Кар...
[1] Score: 0.9444 | Text: Но таким образом ещё усложняется первоначальное мое затрудне...
[2] Score: 0.9793 | Text: Он был женат дважды и имел троих сыновей: старший Дмитрий Фе...
[3] Score: 0.8205 | Text: Брак быстро дал о себе знать. Хотя семейство вскоре примирил...
[4] Score: 0.9568 | Text: Федор Павлович узнал о смерти своей жены, когда был пьян; го...
[5] Score: 0.8226 | Text: Превосходное имение Петра Александровича находилось сразу за...
[6] Score: 0.9276 | Text: Юность и молодость Мити прошли беспорядочно. Он не окончил г...
[7] Score: 0.9123 | Text: Софья Ивановна была из сироток, без родителей с детства. Она...
[8] Score: 0.7971 | Text: После смерти матери с обоими мальчиками случилось то же само...
[9] Score: 0.769 | Text: Ефим Петрович, человек благородный и гуманный, взял на воспи...
[10] Score: 0.8324 | Text: Молодой человек не потерял себя и наше

In [ ]:
import json

# --- 1. ЗАГРУЗКА БЛОКОВ ---
INPUT_FILE = "/kaggle/input/datasets/kirillsilantev/dostoevsky/dostoevsky_blocks_for_neutralization-2.json"
OUTPUT_FILE = "/kaggle/working/dostoevsky_parallel_final_v2.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Формируем промпты для vLLM
prompts = [
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n\nТекст Достоевского:\n{item['original_fragment']}<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    for item in dataset
]

# --- 2. ГЕНЕРАЦИЯ (vLLM автоматически распределит на 2x T4) ---
print(f"Начинаю генерацию для {len(prompts)} фрагментов...")
outputs = llm.generate(prompts, sampling_params)

# Собираем полученные тексты
neutral_texts = [out.outputs[0].text.strip() for out in outputs]
original_texts = [item['original_fragment'] for item in dataset]

# --- 3. ОЦЕНКА СХОДСТВА (BATCH SCORING) ---
print("Вычисляю косинусное сходство через SentenceTransformer...")
# Кодируем тексты пачками (batch_size=32) для максимальной скорости на GPU
emb_orig = embedder.encode(original_texts, batch_size=32, convert_to_tensor=True, show_progress_bar=True)
emb_neut = embedder.encode(neutral_texts, batch_size=32, convert_to_tensor=True, show_progress_bar=True)

# Считаем сходство (попарно по диагонали)
cosine_scores = util.cos_sim(emb_orig, emb_neut).diagonal().tolist()

# --- 4. СБОРКА И СОХРАНЕНИЕ ---
for i, item in enumerate(dataset):
    item['neutral_proxy'] = neutral_texts[i]
    item['topic_fidelity'] = round(cosine_scores[i], 4)
    # Пометим как валидные те, где сходство выше 0.75
    item['is_valid'] = cosine_scores[i] > 0.75

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

avg_score = sum(cosine_scores) / len(cosine_scores)
print(f"\nГотово! Обработано блоков: {len(dataset)}")
print(f"Средний показатель сходства (Topic Fidelity): {avg_score:.4f}")
print(f"Файл сохранен в: {OUTPUT_FILE}")

In [ ]:
INPUT_PATH = '/kaggle/input/datasets/kirillsilantev/dostoevsky-fragments/dostoevsky_original_fragments1.json'
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

def get_prompt(text):
    return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>" \
           f"<|start_header_id|>user<|end_header_id|>\n\nТекст: {text}<|eot_id|>" \
           f"<|start_header_id|>assistant<|end_header_id|>\n\n"

In [ ]:
# На всякий случай определим переменные здесь, если они не подтянулись
TEST_LIMIT = 10

print(f"\n{'='*20} ЭТАП 1: ТЕСТОВЫЙ ЗАПУСК (10 примеров) {'='*20}")

# Берем первые 10 элементов
test_items = dataset[:TEST_LIMIT]

# Используем функцию get_prompt или build_prompt (смотря как ты её назвал выше)
# Если была build_prompt, оставь её. Если get_prompt — замени.
test_prompts = [get_prompt(item['original_fragment']) for item in test_items]

# Генерация
test_outputs = llm.generate(test_prompts, sampling_params)

for i, out in enumerate(test_outputs):
    neutral_text = out.outputs[0].text.strip()
    original_text = test_items[i]['original_fragment']
    
    # Оценка (используем уже загруженный embedder)
    emb1 = embedder.encode(original_text, convert_to_tensor=True)
    emb2 = embedder.encode(neutral_text, convert_to_tensor=True)
    score = torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()
    
    print(f"\n[Пример #{i+1}] Score: {score:.4f}")
    print(f"ORIG: {original_text}...") # Увеличил до 200, чтобы было виднее
    print(f"NEUT: {neutral_text}...")
    print("-" * 50)

In [ ]:
# Если ты используешь ту же ячейку, просто введи 'y' в поле input под результатами теста.
# Или выполни этот блок, если он вынесен отдельно:

print(f"\n{'='*20} ЭТАП 2: ПОЛНАЯ ОБРАБОТКА ({len(dataset)} фрагментов) {'='*20}")

all_prompts = [build_prompt(item['original_fragment']) for item in dataset]
all_outputs = llm.generate(all_prompts, sampling_params)

neutral_texts = [out.outputs[0].text.strip() for out in all_outputs]
originals = [item['original_fragment'] for item in dataset]

print("\nВычисляю финальные метрики Topic Fidelity...")
# Пакетное вычисление эмбеддингов
emb_orig = embedder.encode(originals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)
emb_neut = embedder.encode(neutral_texts, batch_size=32, show_progress_bar=True, convert_to_tensor=True)

scores = torch.nn.functional.cosine_similarity(emb_orig, emb_neut).tolist()

# Сборка финального JSON
for i, item in enumerate(dataset):
    item['neutral_proxy'] = neutral_texts[i]
    item['topic_fidelity'] = round(scores[i], 4)
    item['is_valid'] = scores[i] > 0.75

with open('/kaggle/working/dostoevsky_parallel_final.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)
    
print(f"\nГОТОВО! Файл сохранен: /kaggle/working/dostoevsky_parallel_final.json")

In [ ]:
import pandas as pd

df = pd.read_json('/kaggle/working/dostoevsky_parallel_final.json')

df_clean = df[df['is_valid'] == True].copy()

df_clean = df_clean[df_clean['original_fragment'] != df_clean['neutral_proxy']]

df_clean = df_clean[df_clean['neutral_proxy'].str.len() > 20]

# 4. Сохраняем "Золотой датасет" для обучения
FINAL_TRAIN_PATH = '/kaggle/working/dostoevsky_train_ready.jsonl'
df_clean[['original_fragment', 'neutral_proxy']].to_json(
    FINAL_TRAIN_PATH, orient='records', lines=True, force_ascii=False
)

print(f"Итоговая выборка для обучения LoRA: {len(df_clean)} из {len(df)}")

In [ ]:
# orient='records' создаст список объектов (как в твоем исходном JSON)
# indent=4 сделает файл читаемым (красивым)
df_clean.to_json('/kaggle/working/dostoevsky_parallel_final_LORA.json', 
                 orient='records', 
                 force_ascii=False, 
                 indent=4)

print(f"ГОТОВО! Файл сохранен: /kaggle/working/dostoevsky_parallel_final_LORA.json")

# Дообучение QWEN на перенос стиля Достоевского на существующий текст

In [ ]:
# Полная очистка
!pip uninstall -y unsloth unsloth_zoo trl peft accelerate bitsandbytes

# Установка стабильной релизной версии
!pip install --no-cache-dir "unsloth[colab-new]"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir "trl<0.13.0" peft accelerate bitsandbytes transformers

print("\n--- УСТАНОВКА ЗАВЕРШЕНА! ---")
print("Сделай Restart Session!")

In [ ]:
# Устанавливаем официальные релизные версии
!pip install --no-cache-dir "unsloth[colab-new]"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir "trl<0.13.0" peft accelerate bitsandbytes transformers

In [ ]:
!pip install --no-cache-dir --upgrade "transformers>=4.48.0"

In [ ]:
import torch
import gc
# Очищаем всё, что застряло в памяти после ошибки
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import sys
import os

# --- ХАК ДЛЯ ОБХОДА БАГОВ UNSLOTH ---
try:
    # Создаем фиктивный модуль или правим существующий, чтобы избежать KeyError
    import unsloth_zoo.patching as patching
    if not hasattr(patching, "RL_REPLACEMENTS"):
        patching.RL_REPLACEMENTS = {}
    
    # Затыкаем дыры в словаре
    for key in ["sanitize_logprob", "align_logprobs_with_mask", "grpo_autotune_batch_and_chunks"]:
        if key not in patching.RL_REPLACEMENTS:
            patching.RL_REPLACEMENTS[key] = lambda x: x
    print("Патч словаря применен.")
except:
    # Если модуля еще нет, создаем заглушку в sys.modules
    from types import ModuleType
    m = ModuleType("unsloth_zoo.patching")
    m.RL_REPLACEMENTS = {
        "sanitize_logprob": lambda x: x,
        "align_logprobs_with_mask": lambda x: x,
        "grpo_autotune_batch_and_chunks": lambda x: x
    }
    sys.modules["unsloth_zoo.patching"] = m
    print("Создана системная заглушка для патчей.")

# ТЕПЕРЬ ИМПОРТИРУЕМ
from unsloth import FastLanguageModel
import torch

# --- ИНИЦИАЛИЗАЦИЯ QWEN 2.5 3B ---
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
)

print("Модель готова к обучению!")

In [ ]:
import json
import pandas as pd
from datasets import Dataset

# --- 1. ПОДГОТОВКА ДАННЫХ ---
INPUT_DATA = '/kaggle/input/datasets/kirillsilantev/lora12/dostoevsky_parallel_final_LORA.json'

with open(INPUT_DATA, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Фильтрация
df_train = df[(df['is_valid'] == True) & (df['original_fragment'] != df['neutral_proxy'])].copy()

# СИСТЕМНЫЙ ПРОМПТ (Инструкция)
style_instruction = (
    "Ты — мастер литературной стилизации. Твоя задача — переписать предложенный текст, "
    "используя уникальный слог Ф.М. Достоевского. Сохраняй все факты, но наполни их "
    "характерным психологизмом, сложным синтаксисом и лексикой автора."
)

def formatting_prompts_func(examples):
    instructions = [style_instruction] * len(examples["neutral_proxy"])
    inputs       = examples["neutral_proxy"]
    outputs      = examples["original_fragment"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Формируем структуру ChatML
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n" \
               f"<|im_start|>user\n{input}<|im_end|>\n" \
               f"<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

# Превращаем в Dataset
dataset_hf = Dataset.from_pandas(df_train)

# Очистка колонок
column_names = dataset_hf.column_names
dataset_hf = dataset_hf.map(
    formatting_prompts_func, 
    batched = True,
    remove_columns = column_names 
)

# СОЗДАЕМ ЧИСТЫЙ ДАТАСЕТ (как мы договаривались для обхода ошибок)
final_train_dataset = Dataset.from_dict({"text": dataset_hf["text"]})

print(f"Готово к обучению на {len(final_train_dataset)} примерах.")
print(f"Колонки: {final_train_dataset.column_names}")

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=2048)

print("Токенизация... (это уберет все текстовые ошибки)")
tokenized_dataset = final_train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"] # Удаляем текст, оставляем только числа
)

In [ ]:
from trl import SFTTrainer, SFTConfig

# Настройка максимально аккуратного конфига
sft_config = SFTConfig(
    output_dir = "outputs_dostoevsky",
    max_seq_length = 2048,
    dataset_text_field = "text",
    num_train_epochs = 1,
    per_device_train_batch_size = 2, 
    gradient_accumulation_steps = 8, 
    gradient_checkpointing = True,   # Экономим память
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    optim = "adamw_8bit",
    report_to = "none",
    # Важно: даем трейнеру самому решить, какие колонки удалять
    remove_unused_columns = True, 
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_train_dataset, # Твой датасет с колонкой 'text'
    args = sft_config,
    packing = True, # Этот режим в Unsloth работает стабильнее всего
)

# Принудительно отключаем кэш для обучения (это часто лечит OOM и ошибки лосса)
model.config.use_cache = False

print("\n" + "="*20 + " ЗАПУСК ОБУЧЕНИЯ (STABLE PACKING) " + "="*20)
trainer.train()

# Сохранение
model.save_pretrained_lora("dostoevsky_qwen_lora")
tokenizer.save_pretrained("dostoevsky_qwen_lora")

In [ ]:
# Стандартный способ сохранения для PEFT моделей
model.save_pretrained("dostoevsky_qwen_lora")
tokenizer.save_pretrained("dostoevsky_qwen_lora")

print("Всё! Модель успешно сохранена в папку dostoevsky_qwen_lora")

In [ ]:
from unsloth import FastLanguageModel

# Переключаем в режим предсказания
FastLanguageModel.for_inference(model)

prompt = f"<|im_start|>system\n{style_instruction}<|im_end|>\n<|im_start|>user\nСегодня я зашел в магазин, купил хлеба и почувствовал какую-то странную тоску, глядя на прохожих.<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 150, temperature = 0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
test_cases_deep = [
    "Я стою у порога своего кабака и смотрю на свои старые сапоги, через которые уже видна голая земля. Денег нет совсем, в кармане лишь пустота, а дома ждут голодные дети. Хозяйка вчера кричала на всю лестницу, обещала выкинуть мои пожитки на мостовую. Я чувствую себя последним человеком, ветошкой, об которую каждый волен вытереть ноги.",
    
    "Старик Мармеладов сидел в углу, уронив голову на грязный стол, и тихо всхлипывал. Его шинель превратилась в нечто невообразимое, лохмотья едва прикрывали худые плечи. Вокруг смеялись пьяные мужики, тыкали в него пальцами, а он только смиренно крестился. В его глазах застыла такая бездна отчаяния, что мне стало страшно за свою собственную мелкую душу.",
    
    "Чиновник шел по мосту, прижимая к груди тонкую папку с бумагами, и дрожал от пронизывающего ветра. Он думал о том, что завтра ему нечего будет надеть на службу, а пуговицы на мундире едва держатся. Проходящие мимо господа в богатых экипажах обдавали его грязью из луж, но он даже не поднимал глаз. Вся его жизнь казалась ему случайной ошибкой, коротким сном в холодном петербургском тумане."
]

print("="*40 + " ЛИТЕРАТУРНЫЕ ЧТЕНИЯ " + "="*40)

for i, user_text in enumerate(test_cases_deep, 1):
    # Увеличиваем max_new_tokens, чтобы модель не обрывала мысль
    prompt = f"<|im_start|>system\n{style_instruction}<|im_end|>\n<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 450, 
        temperature = 0.85, 
        top_p = 0.92, 
        repetition_penalty = 1.2,
        do_sample = True
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True).split("assistant\n")[-1]
    
    print(f"\nВАРИАНТ №{i} (ИСХОДНИК):\n{user_text}")
    print(f"\nВАРИАНТ №{i} (СТИЛИЗАЦИЯ):\n{generated_text.strip()}")
    print("-" * 80)